# HW3,  : direct and iterative solvers

We solve the Friedkin-Johnsen equilibrium system

$$(I - \alpha W)\,x = (1 - \alpha)\,s, \qquad A = I - \alpha W, \quad b = (1 - \alpha)\,s$$

on the soc-Epinions1 trust network, with $\alpha = 0.85$.

All the code lives in `hw3_system.py` (system construction) and `solvers.py` (Jacobi, Gauss-Seidel, stopping criteria). This notebook only calls those functions and comments the results. Run it from the repository root.

## 1. System construction and reference solution

**Goal.** Rebuild the matrix $A$ of Member 1 (`01_dataset_preprocessing.ipynb`), check that it matches her numbers, define the right-hand side $b$ and compute a reference solution $x^*$ to validate the iterative solvers against.

**Method.**
- `build_system(alpha)` reads `data/processed/edges_indexed.csv` and repeats Member 1's logic: $W_{ij} = 1/d^{out}_i$ for every edge $i \to j$ (sink rows stay zero), then $A = I - \alpha W$ in CSR format.
- `rcm_permutation(A)` is Member 1's Reverse Cuthill-McKee reordering.
- We verify numerically the properties that guarantee convergence of Jacobi and Gauss-Seidel: unit diagonal, strict diagonal dominance by rows, $\|\alpha W\|_\infty = \alpha$.
- $s_i \sim \mathcal{U}[-1, 1]$ with seed 42 (to be confirmed by the group), saved to `data/processed/m2_s_initial_opinions.npy` so everybody uses the same vector.
- $x^*$ is computed once with the sparse LU factorization `scipy.sparse.linalg.splu` and saved to disk.

In [1]:
import time
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import splu

from hw3_system import build_system, bandwidth, rcm_permutation, initial_opinions, processed_folder

alpha = 0.85

In [2]:
# 1.1 Build W and A = I - alpha W

start = time.perf_counter()
A, W = build_system(alpha)
print(f"build time: {time.perf_counter() - start:.2f} s")

n = A.shape[0]
csr_mb = (A.data.nbytes + A.indices.nbytes + A.indptr.nbytes) / 1024**2

print("shape:", A.shape)
print("nnz(W):", W.nnz, "  nnz(A):", A.nnz, "  (= nnz(W) + n:", W.nnz + n == A.nnz, ")")
print(f"CSR memory of A: {csr_mb:.2f} MB")

build time: 0.03 s
shape: (75879, 75879)
nnz(W): 508837   nnz(A): 584716   (= nnz(W) + n: True )
CSR memory of A: 6.98 MB


In [3]:
# 1.2 Bandwidth before and after RCM, and save the permutation
# RCM breaks ties between nodes of equal degree arbitrarily, so the exact
# permutation can differ between machines. We save ours for reproducibility.

p_path = processed_folder / "m2_rcm_permutation.npy"
if p_path.exists():
    p = np.load(p_path)
else:
    p = rcm_permutation(A)
    np.save(p_path, p)

A_rcm = A[p, :][:, p]
print("Bw(A):", bandwidth(A))
print("Bw(A) after RCM:", bandwidth(A_rcm))

# how many nodes share the same degree in the pattern of A + A^T (RCM ties)
degree = np.diff((A + A.T).tocsr().indptr)
print("nodes with the most common degree:", np.bincount(degree).max(), "of", n)

Bw(A): 75407
Bw(A) after RCM: 44499
nodes with the most common degree: 35755 of 75879


In [4]:
# 1.3 Sanity checks on W and A

row_sums_W = np.asarray(W.sum(axis=1)).ravel()
sink = row_sums_W == 0
print("diag(W) all zero (no self-loops):", np.all(W.diagonal() == 0))
print("row sums of W are 1 or 0:", np.allclose(row_sums_W[~sink], 1.0), "| sink rows:", sink.sum())

diag_A = A.diagonal()
offdiag_abs_sum = np.asarray(abs(A).sum(axis=1)).ravel() - np.abs(diag_A)
print("diag(A) = 1, so D = I:", np.allclose(diag_A, 1.0))
print("max off-diagonal row sum of |A|:", offdiag_abs_sum.max())
print("strictly diagonally dominant by rows:", np.all(np.abs(diag_A) > offdiag_abs_sum))

# Jacobi iteration matrix B_J = D^{-1}(D - A) = alpha W, infinity norm = max row sum
norm_BJ = np.asarray(abs(alpha * W).sum(axis=1)).ravel().max()
print("||B_J||_inf = ||alpha W||_inf:", norm_BJ)
print("A symmetric:", (A - A.T).nnz == 0)
print(f"bound K_inf(A) <= (1 + alpha) / (1 - alpha) = {(1 + alpha) / (1 - alpha):.2f}")

diag(W) all zero (no self-loops): True
row sums of W are 1 or 0: True | sink rows: 15538
diag(A) = 1, so D = I: True
max off-diagonal row sum of |A|: 0.8500000000000016
strictly diagonally dominant by rows: True
||B_J||_inf = ||alpha W||_inf: 0.8500000000000004
A symmetric: False
bound K_inf(A) <= (1 + alpha) / (1 - alpha) = 12.33


In [5]:
# 1.4 Right-hand side b = (1 - alpha) s

s = initial_opinions(n)
b = (1 - alpha) * s
print(f"s: min {s.min():.4f}, max {s.max():.4f}, mean {s.mean():.4f}, std {s.std():.4f}")
print("first 5 entries of s:", np.round(s[:5], 4))

s: min -1.0000, max 1.0000, mean 0.0017, std 0.5768
first 5 entries of s: [ 0.5479 -0.1222  0.7172  0.3947 -0.8116]


In [6]:
# 1.5 Reference solution x* with sparse LU (computed once, then loaded)
# splu wants CSC. permc_spec="COLAMD" reorders the columns to limit fill-in.

x_ref_path = processed_folder / "m2_x_reference_alpha085.npy"
if x_ref_path.exists():
    x_ref = np.load(x_ref_path)
    print("x* loaded from", x_ref_path)
else:
    start = time.perf_counter()
    lu = splu(A.tocsc(), permc_spec="COLAMD")
    lu_time = time.perf_counter() - start
    x_ref = lu.solve(b)
    np.save(x_ref_path, x_ref)

    lu_nnz = lu.L.nnz + lu.U.nnz
    print(f"splu time: {lu_time:.1f} s")
    print(f"nnz(L) + nnz(U): {lu_nnz:,}  fill ratio: {lu_nnz / A.nnz:.1f}x nnz(A)")
    print(f"approx. memory of L and U: {lu_nnz * 12 / 1024**2:,.0f} MB (8 B value + 4 B index per nonzero)")
    del lu

rel_res = np.linalg.norm(b - A @ x_ref, np.inf) / np.linalg.norm(b, np.inf)
print(f"relative residual ||b - A x*||_inf / ||b||_inf = {rel_res:.2e}")

splu time: 148.4 s
nnz(L) + nnz(U): 152,836,025  fill ratio: 261.4x nnz(A)
approx. memory of L and U: 1,749 MB (8 B value + 4 B index per nonzero)
relative residual ||b - A x*||_inf / ||b||_inf = 2.39e-14


In [7]:
# 1.6 Checks on x*: range, and sink rows (row i of A is e_i^T, so x_i = (1 - alpha) s_i)

print(f"x*: min {x_ref.min():.4f}, max {x_ref.max():.4f}, std {x_ref.std():.4f} (std of s: {s.std():.4f})")
print("x* inside [-1, 1]:", np.all(np.abs(x_ref) <= 1))
print("sink nodes: x*_i = (1 - alpha) s_i:", np.allclose(x_ref[sink], (1 - alpha) * s[sink]))

x*: min -0.9516, max 0.8922, std 0.1182 (std of s: 0.5768)
x* inside [-1, 1]: True
sink nodes: x*_i = (1 - alpha) s_i: True


### Results of Section 1

**The system matches Member 1.** $A$ has shape $75{,}879 \times 75{,}879$, $\mathrm{nnz}(A) = 584{,}716 = \mathrm{nnz}(W) + n$ (every diagonal entry is stored, sink rows included) and $Bw(A) = 75{,}407$, as in `01_dataset_preprocessing.ipynb`. In CSR it takes about 7 MB.

**RCM bandwidth.** We obtain $Bw = 44{,}499$ after RCM, while Member 1 reported $45{,}409$. The difference is not an error. In the pattern of $A + A^T$, 35,755 nodes share the same degree, and RCM breaks these ties in an arbitrary order that depends on the sorting routine of the machine. Relabelling the nodes at random before RCM gives bandwidths between about 44,200 and 45,000. In both cases RCM only reduces the bandwidth by about 40%: the hubs of a scale-free graph are connected to nodes all over the matrix, so no ordering can make $A$ narrow-banded. Our permutation is saved in `data/processed/m2_rcm_permutation.npy` so that the Gauss-Seidel experiments (which depend on the ordering) are reproducible.

**Convergence is guaranteed.** The checks confirm the theory:
- $W$ has zero diagonal and row sums equal to 1 (or 0 for the 15,538 sinks), so $\mathrm{diag}(A) = 1$ and $D = I$.
- The off-diagonal row sums of $|A|$ are at most $\alpha = 0.85 < 1 = |a_{ii}|$, so $A$ is strictly diagonally dominant by rows. Jacobi and Gauss-Seidel therefore both converge for any $x^{(0)}$.
- The Jacobi iteration matrix is $B_J = I - A = \alpha W$ with $\|B_J\|_\infty = 0.85$, hence $\rho(B_J) \le 0.85$. The error contracts at least by a factor 0.85 per iteration, so at most about $\log(10^{-8}) / \log(0.85) \approx 113$ iterations are needed for a $10^{-8}$ reduction.
- $A$ is not symmetric, so the SPD convergence theorem for Gauss-Seidel does not apply. Diagonal dominance is the argument we use.
- $K_\infty(A) \le (1 + \alpha)/(1 - \alpha) \approx 12.3$: the system is well conditioned at $\alpha = 0.85$.

**Reference solution.** The sparse LU factorization (`splu` with COLAMD column ordering) gives a relative residual of $2.4 \times 10^{-14}$. With $K_\infty(A) \le 12.3$ the relative error of $x^*$ is at most about $3 \times 10^{-13}$, so $x^*$ is accurate enough to measure the true error of the iterative solvers down to tolerance $10^{-10}$. The factorization is expensive, though: about 150 s and $\mathrm{nnz}(L) + \mathrm{nnz}(U) \approx 1.5 \times 10^8$, i.e. 261 times $\mathrm{nnz}(A)$ (about 1.7 GB instead of 7 MB). This fill-in is studied in Section 5. $x^*$ is saved in `data/processed/m2_x_reference_alpha085.npy`.

**Check of the model.** All equilibrium opinions lie in $[-1, 1]$, as expected since each $x_i$ is a weighted average of the $s_j$ with nonnegative weights summing to at most 1. For sink nodes $x^*_i = (1 - \alpha) s_i = 0.15\, s_i$ exactly. A zero row of $W$ therefore does not mean that a sink "retains 100% of its initial opinion" (as stated in the preprocessing notebook): its opinion is scaled toward 0. If the group wants sinks to keep $s_i$, a self-loop $W_{ii} = 1$ for sinks or $b_i = s_i$ on sink rows would do it. This is a modelling choice for the group and does not affect the convergence analysis, since diagonal dominance holds in all three variants.